# Notebook 4 — Fused MLP on Qwen2.5-3B-Instruct

Port the winning pipeline (ActW pruning + bitmap block-sparse kernel + fused gate/up/act kernel) from the 1B model up to **Gemma 2 2B** (2.6B params, 5.2 GB bf16). Fits comfortably on our 8 GB card with ~1.3 GB headroom for KV cache and activations.

Two questions we want to answer:
1. **Does prefill sparsity speedup scale up at 2B?** At 1B we saw 1.14x at 50% sparsity; the 2B MLP matrices are 2.7x larger (2304×9216 vs 1152×6912), so the matmul is a bigger fraction of total work → sparsity savings should compound.
2. **Does 21% effective sparsity finally break even at 2B?** At 1B it hovered at 0.95–1.03x (noise around break-even). More compute per call should tip it clearly positive.

Also: **bugfix from notebook 3**. Gemma uses `gelu_pytorch_tanh` for the MLP activation, not SiLU. Notebook 3 hardcoded SiLU which caused a small numerical drift. This notebook detects the activation from `model.config.hidden_activation` and uses the right one in both the Triton kernel and the PyTorch fallback.

Scope:
- ActW calibration, build 21% + 50% masks
- MMLU: Dense baseline, BS 21%, Fused 21%
- End-to-end benchmarks: Dense / BS 21% / BS 50% / Fused 21% / Fused 50%

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import triton
import triton.language as tl
import matplotlib.pyplot as plt
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
import time, gc, re, types

import config

# Override model for this notebook only (don't touch config.py)
MODEL_NAME_2B = "Qwen/Qwen2.5-3B-Instruct"

torch.set_float32_matmul_precision("high")
plt.rcParams["figure.dpi"] = 120
print(f"torch: {torch.__version__}  triton: {triton.__version__}")
print(f"device: {torch.cuda.get_device_name(0)}")

TILE_R, TILE_C = config.TILE_SIZES[0]
print(f"tile size: ({TILE_R}, {TILE_C})")

# Sanity: check free VRAM before loading a 5 GB model
free, total = torch.cuda.mem_get_info()
print(f"GPU memory: {free/1e9:.2f} / {total/1e9:.2f} GB free")
assert free > 6e9, "Not enough free VRAM for 2B. Close other kernels and retry."


torch: 2.6.0+cu124  triton: 3.2.0
device: NVIDIA GeForce RTX 4070 Laptop GPU
tile size: (128, 128)
GPU memory: 7.52 / 8.18 GB free


## 1. Load model + detect activation function

`model.config.hidden_activation` tells us which activation the MLP uses. We need the same one in our fused kernel and PyTorch fallback to preserve correctness.

In [2]:
print(f"Loading {MODEL_NAME_2B}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_2B)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_2B,
    torch_dtype=getattr(torch, config.TORCH_DTYPE),
    device_map=config.DEVICE,
)
model.eval()
dtype = getattr(torch, config.TORCH_DTYPE)

print(f"params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"peak GPU after load: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

# Architecture info
cfg = model.config
print(f"\nArchitecture:")
print(f"  hidden_size:       {cfg.hidden_size}")
print(f"  num_layers:        {cfg.num_hidden_layers}")
print(f"  intermediate_size: {cfg.intermediate_size}")
print(f"  num_heads:         {getattr(cfg, 'num_attention_heads', '?')}")
print(f"  head_dim:          {getattr(cfg, 'head_dim', '?')}")
# Qwen uses hidden_act, Gemma uses hidden_activation
HIDDEN_ACT = getattr(cfg, "hidden_activation", None) or getattr(cfg, "hidden_act", None)
assert HIDDEN_ACT is not None, f"could not find activation in config; available: {dir(cfg)}"
print(f"  activation:        {HIDDEN_ACT}")

# PyTorch fallback for this activation (used in hybrid-dispatch path)
def torch_act(x):
    if HIDDEN_ACT == "silu":
        return F.silu(x)
    elif HIDDEN_ACT == "gelu_pytorch_tanh":
        return F.gelu(x, approximate="tanh")
    elif HIDDEN_ACT == "gelu":
        return F.gelu(x)
    else:
        raise ValueError(f"unsupported activation: {HIDDEN_ACT}")

# Integer code for Triton constexpr dispatch (0=silu, 1=gelu_pytorch_tanh, 2=gelu_exact)
ACT_CODE = {"silu": 0, "gelu_pytorch_tanh": 1, "gelu": 2}[HIDDEN_ACT]
print(f"  ACT_CODE (kernel): {ACT_CODE}")


Loading Qwen/Qwen2.5-3B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

params: 3.09B
peak GPU after load: 6.17 GB

Architecture:
  hidden_size:       2048
  num_layers:        36
  intermediate_size: 11008
  num_heads:         16
  head_dim:          ?
  activation:        silu
  ACT_CODE (kernel): 0


## 2. ActW calibration + build masks

Same recipe as the 1B notebook: per-component-type z-score, 30% global target + 50% per-matrix cap for the realistic config (21% effective), plus 50% per-matrix uniform for the kernel-speed sanity check.

In [3]:
from datasets import load_dataset
from tqdm import tqdm

def is_mlp(name):
    return any(k in name for k in config.PRUNE_TARGETS_PATTERNS)

def get_component_type(name):
    for p in config.PRUNE_TARGETS_PATTERNS:
        if p in name:
            return p
    return "unknown"

def compute_tile_frob(W, tr=TILE_R, tc=TILE_C):
    W = W.detach().float().cpu()
    out_dim, in_dim = W.shape
    nr, nc = out_dim // tr, in_dim // tc
    Wt = W[:nr*tr, :nc*tc].reshape(nr, tr, nc, tc).permute(0, 2, 1, 3)
    return Wt.reshape(nr, nc, -1).norm(dim=-1).numpy()

print("loading calibration data...")
cal_ds = load_dataset(
    config.CALIBRATION_DATASET,
    config.CALIBRATION_SUBSET,
    split="train",
    streaming=True,
    trust_remote_code=True,
)
cal_texts = []
for i, ex in enumerate(cal_ds):
    if i >= config.CALIBRATION_SAMPLES:
        break
    t = ex.get("text", "")
    if len(t) > 50:
        cal_texts.append(t)
cal_encodings = [
    tokenizer(t, return_tensors="pt", max_length=config.CALIBRATION_SEQ_LEN, truncation=True)
    for t in cal_texts
]
print(f"loaded {len(cal_encodings)} samples")

loading calibration data...


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

loaded 1022 samples


In [4]:
# Frobenius tile norms for all MLP weights
importance_frob = {}
for name, param in model.named_parameters():
    if param.ndim != 2 or not is_mlp(name):
        continue
    importance_frob[name] = compute_tile_frob(param)
print(f"frob scored: {len(importance_frob)} MLP matrices")

# Activation hooks
activation_stats = {}
hooks = []
def make_hook(pname, in_dim):
    nc = in_dim // TILE_C
    def h(mod, inp, out):
        x = inp[0].detach().float().reshape(-1, inp[0].shape[-1])
        x = x[:, :nc*TILE_C].reshape(x.shape[0], nc, TILE_C)
        cn = x.norm(dim=-1).mean(dim=0).cpu()
        if pname not in activation_stats:
            activation_stats[pname] = (cn, 1)
        else:
            s, c = activation_stats[pname]

            activation_stats[pname] = (s + cn, c + 1)
    return h

for name, module in model.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    pn = name + ".weight"
    if not is_mlp(pn) or module.in_features < TILE_C:
        continue
    hooks.append(module.register_forward_hook(make_hook(pn, module.in_features)))
print(f"registered {len(hooks)} hooks")

print(f"calibrating on {len(cal_encodings)} samples (forward only)...")
with torch.no_grad():
    for enc in tqdm(cal_encodings, desc="calib"):
        model(**{k: v.to(config.DEVICE) for k, v in enc.items()})
for h in hooks:
    h.remove()
print(f"activation stats collected for {len(activation_stats)} matrices")

frob scored: 108 MLP matrices
registered 108 hooks
calibrating on 1022 samples (forward only)...


calib: 100%|█████████████████████████████████████████████████████| 1022/1022 [01:26<00:00, 11.82it/s]

activation stats collected for 108 matrices


In [5]:
# Activation-weighted + per-component-type z-score
importance_actw = {}
for name, fmap in importance_frob.items():
    if name not in activation_stats:
        continue
    cn, count = activation_stats[name]
    importance_actw[name] = fmap * (cn / count).numpy()[np.newaxis, :]

comp_stats = {}
for ct in config.PRUNE_TARGETS_PATTERNS:
    vals = np.concatenate([m.ravel() for n, m in importance_actw.items() if ct in n])
    comp_stats[ct] = (vals.mean(), vals.std())
    print(f"  {ct}: mean={vals.mean():.3f} std={vals.std():.3f} tiles={len(vals)}")

importance_actw_norm = {}
for name, m in importance_actw.items():
    mu, sd = comp_stats[get_component_type(name)]
    importance_actw_norm[name] = (m - mu) / sd if sd > 1e-8 else np.zeros_like(m)

# 30% target with 50% per-matrix cap => ~21% effective
PRUNE_RATIO = 0.30
all_scores = np.concatenate([m.ravel() for m in importance_actw_norm.values()])
threshold = float(np.percentile(all_scores, PRUNE_RATIO * 100))
def capped_mask(norm_map, thresh, max_prune=config.MAX_PRUNE_PER_MATRIX):
    gm = norm_map < thresh
    if gm.mean() <= max_prune:
        return gm
    local = float(np.percentile(norm_map.ravel(), max_prune * 100))
    return norm_map < local
prune_masks_21 = {name: capped_mask(m, threshold) for name, m in importance_actw_norm.items()}

# 50% uniform per matrix
prune_masks_50 = {}
for name, norm_map in importance_actw_norm.items():
    local = float(np.percentile(norm_map.ravel(), 50))
    prune_masks_50[name] = norm_map < local

def eff_sparsity(masks):
    total = sum(m.size for m in masks.values())
    pruned = sum(m.sum() for m in masks.values())
    return pruned, total, pruned / total

p21, t21, r21 = eff_sparsity(prune_masks_21)
p50, t50, r50 = eff_sparsity(prune_masks_50)
print(f"\n21% ActW (capped): {p21}/{t21} = {100*r21:.1f}% effective")
print(f"50% uniform:       {p50}/{t50} = {100*r50:.1f}% effective")

  gate_proj: mean=29.675 std=32.763 tiles=49536
  up_proj: mean=28.872 std=28.909 tiles=49536
  down_proj: mean=8.571 std=11.013 tiles=49536

21% ActW (capped): 31008/148608 = 20.9% effective
50% uniform:       74304/148608 = 50.0% effective


## 2b. Gradient / Taylor calibration (extra scoring method)

Taylor importance: `importance(tile) = Σ |w * ∂L/∂w|`. Captures weight size, input activation, AND downstream loss sensitivity in one metric.

On a 3B model this is memory-tight. Workarounds:
- Freeze non-MLP params → no gradients allocated for attention
- Enable gradient checkpointing → minimal saved activations
- Per-parameter post-accumulate hook → compute Taylor + free grad immediately (peak grad memory = one layer's MLP, not all 36)
- Short seq_len (256) and 256 samples (half of ActW) to bound runtime

In [6]:
# Gradient/Taylor calibration — memory-efficient for 3B on 8 GB
print("Preparing gradient calibration...")

# 1. Freeze all params, unfreeze MLP 2D weights only
for param in model.parameters():
    param.requires_grad_(False)
n_trainable = 0
for name, param in model.named_parameters():
    if is_mlp(name) and param.ndim == 2:
        param.requires_grad_(True)
        n_trainable += 1
print(f"  trainable MLP params: {n_trainable}")

# 2. Gradient checkpointing
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
model.config.use_cache = False

# 3. Accumulate Taylor importance on GPU (tiny buffers, one per matrix)
importance_grad_gpu = {}

def make_post_grad_hook(pname):
    def hook(param):
        if param.grad is None:
            return
        taylor = (param.data * param.grad).abs().float()  # fp32 on GPU
        out_dim, in_dim = taylor.shape
        nr, nc = out_dim // TILE_R, in_dim // TILE_C
        tt = taylor[:nr*TILE_R, :nc*TILE_C].reshape(nr, TILE_R, nc, TILE_C).permute(0, 2, 1, 3)
        tile_scores = tt.reshape(nr, nc, -1).sum(dim=-1)
        if pname in importance_grad_gpu:
            importance_grad_gpu[pname] += tile_scores
        else:
            importance_grad_gpu[pname] = tile_scores.clone()
        # Free the gradient immediately — critical for memory
        param.grad = None
    return hook

hook_handles = []
for name, param in model.named_parameters():
    if is_mlp(name) and param.ndim == 2 and param.requires_grad:
        h = param.register_post_accumulate_grad_hook(make_post_grad_hook(name))
        hook_handles.append(h)

# 4. Short-seq calibration subset
N_GRAD_SAMPLES = 256
SEQ_LEN_GRAD = 128  # halved from 256 to shrink logits tensor (Qwen vocab is 152k)
grad_cal_encodings = [
    tokenizer(t, return_tensors="pt", max_length=SEQ_LEN_GRAD, truncation=True)
    for t in cal_texts[:N_GRAD_SAMPLES]
]
print(f"  {len(grad_cal_encodings)} samples, seq_len={SEQ_LEN_GRAD}")

# 5. Run calibration
print("running gradient calibration (forward + backward)...")
n_grad_samples = 0
for enc in tqdm(grad_cal_encodings, desc="grad calib"):
    model.zero_grad(set_to_none=True)
    inputs = {k: v.to(config.DEVICE) for k, v in enc.items()}
    outputs = model(**inputs, labels=inputs["input_ids"])
    outputs.loss.backward()
    n_grad_samples += 1

# 6. Clean up
for h in hook_handles:
    h.remove()
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.config.use_cache = True
for param in model.parameters():
    param.requires_grad_(False)

# 7. Move accumulator to CPU as numpy, average
importance_grad = {name: (v / n_grad_samples).cpu().numpy()
                   for name, v in importance_grad_gpu.items()}
del importance_grad_gpu
gc.collect()
torch.cuda.empty_cache()
print(f"Taylor importance scored for {len(importance_grad)} matrices ({n_grad_samples} samples)")
print(f"  peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


Preparing gradient calibration...
  trainable MLP params: 108
  256 samples, seq_len=128
running gradient calibration (forward + backward)...


grad calib: 100%|██████████████████████████████████████████████████| 256/256 [01:49<00:00,  2.33it/s]


Taylor importance scored for 108 matrices (256 samples)
  peak GPU: 7.01 GB


In [7]:
# Build Taylor-based masks (30% target + 50% per-matrix cap, same as ActW)
comp_stats_grad = {}
for ct in config.PRUNE_TARGETS_PATTERNS:
    vals = np.concatenate([m.ravel() for n, m in importance_grad.items() if ct in n])
    comp_stats_grad[ct] = (vals.mean(), vals.std())
    print(f"  {ct} (grad): mean={vals.mean():.4f} std={vals.std():.4f}")

importance_grad_norm = {}
for name, m in importance_grad.items():
    mu, sd = comp_stats_grad[get_component_type(name)]
    importance_grad_norm[name] = (m - mu) / sd if sd > 1e-8 else np.zeros_like(m)

all_scores_g = np.concatenate([m.ravel() for m in importance_grad_norm.values()])
threshold_grad = float(np.percentile(all_scores_g, 30))
prune_masks_grad_21 = {name: capped_mask(m, threshold_grad) for name, m in importance_grad_norm.items()}

p_g, t_g, r_g = eff_sparsity(prune_masks_grad_21)
print(f"\n21% Taylor (capped): {p_g}/{t_g} = {100*r_g:.1f}% effective sparsity")


  gate_proj (grad): mean=0.0387 std=0.0122
  up_proj (grad): mean=0.0478 std=0.0156
  down_proj (grad): mean=0.0522 std=0.0158

21% Taylor (capped): 27758/148608 = 18.7% effective sparsity


## 3. Bitmap block-sparse kernel + `BlockSparseLinear` (hybrid dispatch)

Same kernel as notebook 3. The `SMALL_M_THRESHOLD=64` dispatch keeps decode on dense cuBLAS.

In [8]:
@triton.jit
def block_sparse_matmul_kernel(
    X_ptr, W_ptr, Y_ptr, bitmap_ptr,
    M, N, K,
    stride_xm, stride_xk,
    stride_wn, stride_wk,
    stride_ym, stride_yn,
    stride_bn, stride_bk,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    x_ptrs = X_ptr + offs_m[:, None] * stride_xm + offs_k[None, :] * stride_xk
    w_ptrs = W_ptr + offs_n[:, None] * stride_wn + offs_k[None, :] * stride_wk
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    n_k_tiles = tl.cdiv(K, BLOCK_K)
    for k_tile in range(n_k_tiles):
        nz = tl.load(bitmap_ptr + pid_n * stride_bn + k_tile * stride_bk)
        if nz != 0:
            k_start = k_tile * BLOCK_K
            x_mask = (offs_m[:, None] < M) & ((offs_k[None, :] + k_start) < K)
            w_mask = (offs_n[:, None] < N) & ((offs_k[None, :] + k_start) < K)
            x = tl.load(x_ptrs, mask=x_mask, other=0.0)
            w = tl.load(w_ptrs, mask=w_mask, other=0.0)
            acc += tl.dot(x, tl.trans(w))
        x_ptrs += BLOCK_K * stride_xk
        w_ptrs += BLOCK_K * stride_wk
    y_ptrs = Y_ptr + offs_m[:, None] * stride_ym + offs_n[None, :] * stride_yn
    y_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(y_ptrs, acc.to(Y_ptr.dtype.element_ty), mask=y_mask)


def block_sparse_matmul(x, w, bitmap, block_m=64, block_n=TILE_R, block_k=TILE_C):
    M, K = x.shape
    N, _ = w.shape
    y = torch.empty((M, N), device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    block_sparse_matmul_kernel[grid](
        x, w, y, bitmap,
        M, N, K,
        x.stride(0), x.stride(1),
        w.stride(0), w.stride(1),
        y.stride(0), y.stride(1),
        bitmap.stride(0), bitmap.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k,
    )
    return y


class BlockSparseLinear(nn.Module):
    SMALL_M_THRESHOLD = 64

    def __init__(self, in_features, out_features, bitmap, weight, bias=None,
                 tile_r=TILE_R, tile_c=TILE_C, small_m_threshold=None):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.tile_r = tile_r
        self.tile_c = tile_c
        self.small_m_threshold = (small_m_threshold if small_m_threshold is not None
                                  else self.SMALL_M_THRESHOLD)
        self.register_buffer("weight", weight.contiguous())
        self.register_buffer("bitmap", bitmap.contiguous())
        if bias is not None:
            self.register_buffer("bias", bias.contiguous())
        else:
            self.bias = None

    def forward(self, x):
        orig_shape = x.shape
        x2 = x.reshape(-1, orig_shape[-1])
        M = x2.shape[0]
        if M < self.small_m_threshold:
            y = x2 @ self.weight.T
        else:
            if not x2.is_contiguous():
                x2 = x2.contiguous()
            y = block_sparse_matmul(x2, self.weight, self.bitmap,
                                    block_n=self.tile_r, block_k=self.tile_c)
        if self.bias is not None:
            y = y + self.bias
        return y.reshape(*orig_shape[:-1], self.out_features)

    @classmethod
    def from_dense(cls, linear, prune_mask, tile_r=TILE_R, tile_c=TILE_C):
        out_f, in_f = linear.weight.shape
        nr, nc = out_f // tile_r, in_f // tile_c
        keep = (~prune_mask[:nr, :nc]).astype(np.int8)
        bitmap = torch.from_numpy(keep).to(linear.weight.device)
        w = linear.weight.detach().clone()
        for i in range(nr):
            for j in range(nc):
                if prune_mask[i, j]:
                    w[i*tile_r:(i+1)*tile_r, j*tile_c:(j+1)*tile_c] = 0
        b = linear.bias.detach().clone() if linear.bias is not None else None
        return cls(in_f, out_f, bitmap, w, b, tile_r, tile_c)

print("BlockSparseLinear (hybrid dispatch) ready.")

BlockSparseLinear (hybrid dispatch) ready.


## 4. Activation-aware fused kernel

The kernel does two block-sparse matmuls (gate, up) in one grid, then applies `act(acc_g) * acc_u` elementwise. Activation is selected via `ACT_FN: tl.constexpr` — compiles a different version per activation, no runtime branch cost.

- `ACT_FN=0`: SiLU (`x * sigmoid(x)`)
- `ACT_FN=1`: GELU pytorch_tanh approximation (`0.5 * x * (1 + tanh(sqrt(2/π) * (x + 0.044715 * x³)))`)
- `ACT_FN=2`: Exact GELU via erf

In [9]:
SQRT_2_OVER_PI = 0.7978845608028654
GELU_TANH_COEF = 0.044715

@triton.jit
def fused_gate_up_act_kernel(
    X_ptr, Wg_ptr, Wu_ptr, Bg_ptr, Bu_ptr, H_ptr,
    M, N, K,
    stride_xm, stride_xk,
    stride_wgn, stride_wgk,
    stride_wun, stride_wuk,
    stride_hm, stride_hn,
    stride_bgn, stride_bgk,
    stride_bun, stride_buk,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    ACT_FN: tl.constexpr,  # 0=silu, 1=gelu_pytorch_tanh, 2=gelu_exact
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    x_ptrs  = X_ptr  + offs_m[:, None] * stride_xm  + offs_k[None, :] * stride_xk
    wg_ptrs = Wg_ptr + offs_n[:, None] * stride_wgn + offs_k[None, :] * stride_wgk
    wu_ptrs = Wu_ptr + offs_n[:, None] * stride_wun + offs_k[None, :] * stride_wuk

    acc_g = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    acc_u = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    n_k_tiles = tl.cdiv(K, BLOCK_K)
    for k_tile in range(n_k_tiles):
        nz_g = tl.load(Bg_ptr + pid_n * stride_bgn + k_tile * stride_bgk)
        nz_u = tl.load(Bu_ptr + pid_n * stride_bun + k_tile * stride_buk)
        if (nz_g != 0) | (nz_u != 0):
            k_start = k_tile * BLOCK_K
            x_mask = (offs_m[:, None] < M) & ((offs_k[None, :] + k_start) < K)
            x = tl.load(x_ptrs, mask=x_mask, other=0.0)
            wnk_mask = (offs_n[:, None] < N) & ((offs_k[None, :] + k_start) < K)
            if nz_g != 0:
                wg = tl.load(wg_ptrs, mask=wnk_mask, other=0.0)
                acc_g += tl.dot(x, tl.trans(wg))
            if nz_u != 0:
                wu = tl.load(wu_ptrs, mask=wnk_mask, other=0.0)
                acc_u += tl.dot(x, tl.trans(wu))
        x_ptrs  += BLOCK_K * stride_xk
        wg_ptrs += BLOCK_K * stride_wgk
        wu_ptrs += BLOCK_K * stride_wuk

    # Activation on acc_g, then * acc_u
    if ACT_FN == 0:
        # SiLU: x * sigmoid(x)
        act_g = acc_g * tl.sigmoid(acc_g)
    elif ACT_FN == 1:
        # gelu_pytorch_tanh: 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
        x3 = acc_g * acc_g * acc_g
        inner = SQRT_2_OVER_PI * (acc_g + GELU_TANH_COEF * x3)
        # tanh(x) = 2*sigmoid(2x) - 1  (portable, avoids tl.math.tanh)
        tanh_inner = 2.0 * tl.sigmoid(2.0 * inner) - 1.0
        act_g = 0.5 * acc_g * (1.0 + tanh_inner)
    else:
        # Exact GELU: 0.5 * x * (1 + erf(x / sqrt(2)))
        # Exact GELU disabled in Triton 3.2 (no tl.math.erf). If you need it, approximate via tanh variant (ACT_FN=1).
        act_g = 0.5 * acc_g * (1.0 + (2.0 * tl.sigmoid(2.0 * acc_g * 0.7071067811865475 * 1.5957691216057308) - 1.0))

    h = act_g * acc_u

    h_ptrs = H_ptr + offs_m[:, None] * stride_hm + offs_n[None, :] * stride_hn
    h_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(h_ptrs, h.to(H_ptr.dtype.element_ty), mask=h_mask)


def fused_gate_up_act(x, w_gate, w_up, bitmap_gate, bitmap_up,
                      act_code: int,
                      block_m=64, block_n=TILE_R, block_k=TILE_C):
    """x: (M, K), w_gate/up: (N, K) -> h: (M, N) = act(x @ w_gate.T) * (x @ w_up.T)"""
    assert x.shape[1] == w_gate.shape[1] == w_up.shape[1]
    assert w_gate.shape == w_up.shape
    M, K = x.shape
    N, _ = w_gate.shape
    h = torch.empty((M, N), device=x.device, dtype=x.dtype)
    grid = (triton.cdiv(M, block_m), triton.cdiv(N, block_n))
    fused_gate_up_act_kernel[grid](
        x, w_gate, w_up, bitmap_gate, bitmap_up, h,
        M, N, K,
        x.stride(0), x.stride(1),
        w_gate.stride(0), w_gate.stride(1),
        w_up.stride(0), w_up.stride(1),
        h.stride(0), h.stride(1),
        bitmap_gate.stride(0), bitmap_gate.stride(1),
        bitmap_up.stride(0), bitmap_up.stride(1),
        BLOCK_M=block_m, BLOCK_N=block_n, BLOCK_K=block_k,
        ACT_FN=act_code,
    )
    return h

print(f"fused_gate_up_act kernel ready (compiling for ACT_CODE={ACT_CODE} on first call)")

fused_gate_up_act kernel ready (compiling for ACT_CODE=0 on first call)


In [10]:
# Correctness test on the ACTUAL activation the model uses
torch.manual_seed(0)
Mt, Nt, Kt = 256, 1024, 1152
xt = torch.randn(Mt, Kt, device="cuda", dtype=dtype)
wgt = torch.randn(Nt, Kt, device="cuda", dtype=dtype) * 0.1
wut = torch.randn(Nt, Kt, device="cuda", dtype=dtype) * 0.1

pm_g = (np.random.rand(Nt // TILE_R, Kt // TILE_C) > 0.70).astype(np.int8)
pm_u = (np.random.rand(Nt // TILE_R, Kt // TILE_C) > 0.70).astype(np.int8)
bg = torch.from_numpy(pm_g).to("cuda").contiguous()
bu = torch.from_numpy(pm_u).to("cuda").contiguous()

wg_zero = wgt.clone()
wu_zero = wut.clone()
for i in range(Nt // TILE_R):
    for j in range(Kt // TILE_C):
        if pm_g[i, j] == 0:
            wg_zero[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0
        if pm_u[i, j] == 0:
            wu_zero[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0

# Reference uses model's actual activation
gate_ref = xt @ wg_zero.T
up_ref   = xt @ wu_zero.T
h_ref = torch_act(gate_ref) * up_ref

h_fused = fused_gate_up_act(xt, wg_zero, wu_zero, bg, bu, ACT_CODE)

diff = (h_ref - h_fused).abs()
print(f"activation: {HIDDEN_ACT}")
print(f"max abs diff:  {diff.max().item():.5f}")
print(f"mean abs diff: {diff.mean().item():.6f}")
print(f"|h_ref| mean:  {h_ref.abs().mean().item():.4f}")
max_d = diff.max().item()
mean_d = diff.mean().item()
h_max = h_ref.abs().max().item()
rel = max_d / max(h_max, 1e-6)
print(f"max/|h|: {rel:.4f}  (relative tolerance)")
# Max is a bf16-rounding outlier tail. Mean is the real correctness signal.
# Fused uses FP32 throughout; reference round-trips bf16. Tolerate max up to 2.0 so long as mean is tiny and relative is tight.
assert mean_d < 0.01, f"fused kernel mean diff too high: {mean_d}"
assert rel < 0.15, f"fused kernel relative diff too high: {rel}"
print("fused kernel correctness: OK  (mean diff and relative diff within tolerance)")

activation: silu
max abs diff:  0.25000
mean abs diff: 0.001778
|h_ref| mean:  0.7188
max/|h|: 0.0067  (relative tolerance)
fused kernel correctness: OK  (mean diff and relative diff within tolerance)


## 5. Apply fused forward to the model

In [11]:
FUSE_M_THRESHOLD = 64

def fused_mlp_forward(self, x):
    """Monkey-patched MLP forward: fused gate+up+act at M>=64, regular path at M<64."""
    orig_shape = x.shape
    x2 = x.reshape(-1, orig_shape[-1])
    M = x2.shape[0]
    if M < FUSE_M_THRESHOLD:
        gate = self.gate_proj(x)
        up   = self.up_proj(x)
        h = torch_act(gate) * up
        return self.down_proj(h)
    if not x2.is_contiguous():
        x2 = x2.contiguous()
    h_flat = fused_gate_up_act(
        x2,
        self.gate_proj.weight, self.up_proj.weight,
        self.gate_proj.bitmap, self.up_proj.bitmap,
        ACT_CODE,
    )
    h = h_flat.reshape(*orig_shape[:-1], h_flat.shape[-1])
    return self.down_proj(h)


# ============================================================
# Load-once architecture: keep the calibration model around,
# cache original MLP weights on CPU, and provide configure_model()
# to switch variants in-place (no reloading, no memory spikes).
# ============================================================

print("Caching original MLP weights to CPU (for per-variant restore)...")
original_mlp_weights = {}
for name, param in model.named_parameters():
    if is_mlp(name) and param.ndim == 2:
        original_mlp_weights[name] = param.data.cpu().clone()
cpu_gb = sum(w.numel() * w.element_size() for w in original_mlp_weights.values()) / 1e9
print(f"  {len(original_mlp_weights)} weights cached, {cpu_gb:.2f} GB on CPU")

# Index Linear modules by parameter name
mlp_linear_modules = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        pn = name + ".weight"
        if pn in original_mlp_weights:
            mlp_linear_modules[pn] = (name, module)

# Find MLP wrapper modules (gate+up+down containers)
mlp_wrapper_modules = []
for _, module in model.named_modules():
    if (hasattr(module, "gate_proj") and hasattr(module, "up_proj")
        and hasattr(module, "down_proj")
        and isinstance(module.gate_proj, nn.Linear)):
        mlp_wrapper_modules.append(module)
print(f"  {len(mlp_wrapper_modules)} MLP wrapper modules indexed")


def bs_linear_forward(self, x):
    """Drop-in forward for nn.Linear: block-sparse kernel at M>=64, dense fallback at M<64."""
    orig_shape = x.shape
    x2 = x.reshape(-1, orig_shape[-1])
    M = x2.shape[0]
    if M < 64:
        y = F.linear(x2, self.weight, self.bias)
    else:
        if not x2.is_contiguous():
            x2 = x2.contiguous()
        y = block_sparse_matmul(x2, self.weight, self.bitmap)
        if self.bias is not None:
            y = y + self.bias
    return y.reshape(*orig_shape[:-1], self.out_features)


def _apply_masks_inplace(mask_dict):
    """Zero pruned tiles in-place on each MLP weight; attach `bitmap` buffer."""
    for pname, (_, module) in mlp_linear_modules.items():
        mask = mask_dict[pname]
        w = module.weight.data
        out_f, in_f = w.shape
        nr, nc = out_f // TILE_R, in_f // TILE_C
        for i in range(nr):
            for j in range(nc):
                if mask[i, j]:
                    w[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0
        keep = (~mask[:nr, :nc]).astype(np.int8)
        module.bitmap = torch.from_numpy(keep).to(w.device)


def _restore_original_weights():
    """Copy pristine weights from CPU cache back to each Linear."""
    with torch.no_grad():
        for pname, (_, module) in mlp_linear_modules.items():
            module.weight.data.copy_(original_mlp_weights[pname].to(module.weight.device, non_blocking=True))


def _unpatch_forwards():
    """Remove monkey-patched forward methods and bitmap buffers."""
    for _, (_, module) in mlp_linear_modules.items():
        if "forward" in module.__dict__:
            del module.__dict__["forward"]
        if hasattr(module, "bitmap"):
            del module.bitmap
    for mlp in mlp_wrapper_modules:
        if "forward" in mlp.__dict__:
            del mlp.__dict__["forward"]


def configure_model(variant: str, mask_dict=None):
    """Switch the shared `model` to a variant in-place.
    variant: 'dense' | 'bs' | 'fused'
    mask_dict: required for 'bs' / 'fused' — pruning masks keyed by parameter name.
    """
    assert variant in ("dense", "bs", "fused"), f"unknown variant: {variant}"
    if variant != "dense":
        assert mask_dict is not None, "bs/fused variants need mask_dict"

    # 1) Restore pristine dense weights (undo any in-place zeroing from previous variant)
    _restore_original_weights()
    # 2) Remove any forward patches + bitmap buffers from previous variant
    _unpatch_forwards()
    # 3) Apply variant
    if variant != "dense":
        _apply_masks_inplace(mask_dict)
        for _, (_, module) in mlp_linear_modules.items():
            module.forward = types.MethodType(bs_linear_forward, module)
        if variant == "fused":
            for mlp in mlp_wrapper_modules:
                mlp.forward = types.MethodType(fused_mlp_forward, mlp)

    torch.cuda.empty_cache()
    alloc = torch.cuda.memory_allocated() / 1e9
    print(f"  configured '{variant}' | pytorch allocated: {alloc:.2f} GB")


print("\nconfigure_model() ready — call with 'dense' / 'bs' / 'fused' to switch in-place.")


Caching original MLP weights to CPU (for per-variant restore)...
  108 weights cached, 4.87 GB on CPU
  36 MLP wrapper modules indexed

configure_model() ready — call with 'dense' / 'bs' / 'fused' to switch in-place.


In [12]:
# Deep GPU memory leak diagnostic
# Walks ALL Python objects (not just globals) to find every live CUDA tensor.
# Then traces back who is holding each big one.
import sys, gc

# Clear IPython's stored exception traceback (holds frame references!)
for attr in ["last_traceback", "last_value", "last_type", "last_exc"]:
    if hasattr(sys, attr):
        try:
            setattr(sys, attr, None)
        except Exception:
            pass

# Clear IPython output history if present
try:
    ip = get_ipython()
    ip.history_manager.reset()
    ip.displayhook.flush()
    for name in ["_", "__", "___", "_i", "_ii", "_iii"]:
        if name in ip.user_ns:
            ip.user_ns[name] = None
    # Clear the numbered output cache
    if hasattr(ip, "user_ns") and "Out" in ip.user_ns:
        ip.user_ns["Out"].clear()
    if hasattr(ip, "user_ns") and "_oh" in ip.user_ns:
        ip.user_ns["_oh"].clear()
except NameError:
    pass

gc.collect()
try:
    torch.cuda.synchronize()
except Exception:
    pass
torch.cuda.empty_cache()

# Scan EVERY python object for cuda tensors
cuda_tensors = []
for obj in gc.get_objects():
    try:
        if isinstance(obj, torch.Tensor) and obj.is_cuda:
            size_mb = obj.numel() * obj.element_size() / 1e6
            if size_mb > 1:
                cuda_tensors.append((size_mb, tuple(obj.shape), obj.dtype, id(obj)))
    except (ReferenceError, RuntimeError):
        pass

cuda_tensors.sort(reverse=True)
total = sum(t[0] for t in cuda_tensors)
print(f"=== Deep scan: {len(cuda_tensors)} CUDA tensors > 1MB, total {total:.0f} MB ===")
print(f"{'size':>10} {'shape':<30} {'dtype':<20} {'referrers':<40}")
print("-" * 100)

# For the top 15, trace what refers to them
for sz, shape, dt, oid in cuda_tensors[:15]:
    # Get the tensor back from id — find it in objects
    tensor = None
    for obj in gc.get_objects():
        if id(obj) == oid:
            tensor = obj
            break
    refs = []
    if tensor is not None:
        for ref in gc.get_referrers(tensor):
            ref_type = type(ref).__name__
            if ref_type in ("frame", "list", "tuple", "dict", "set"):
                # Try to find what names reference it
                if ref_type == "dict":
                    try:
                        items_snapshot = list(ref.items())
                    except RuntimeError:
                        items_snapshot = []
                    for k, v in items_snapshot:
                        if v is tensor:
                            refs.append(f"dict[{k!r}]")
                            break
                    else:
                        refs.append(f"dict(size={len(ref)})")
                elif ref_type == "list":
                    refs.append(f"list(len={len(ref)})")
                else:
                    refs.append(ref_type)
            else:
                refs.append(ref_type)
            if len(refs) >= 3:
                break
    print(f"{sz:>7.1f}MB  {str(shape):<30} {str(dt):<20} {', '.join(refs[:3])}")

alloc = torch.cuda.memory_allocated()/1e9
print(f"\npytorch allocated now: {alloc:.2f} GB")


/tmp/ipykernel_69180/4290972918.py:41: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, torch.Tensor) and obj.is_cuda:


=== Deep scan: 258 CUDA tensors > 1MB, total 6220 MB ===
      size shape                          dtype                referrers                               
----------------------------------------------------------------------------------------------------
  622.3MB  (151936, 2048)                 torch.bfloat16       dict['obj'], dict['weight'], dict['weight']
   45.1MB  (11008, 2048)                  torch.bfloat16       dict['obj'], dict['weight']
   45.1MB  (11008, 2048)                  torch.bfloat16       dict['obj'], dict['weight']
   45.1MB  (11008, 2048)                  torch.bfloat16       dict['obj'], dict['weight']
   45.1MB  (11008, 2048)                  torch.bfloat16       dict['obj'], dict['weight']
   45.1MB  (11008, 2048)                  torch.bfloat16       dict['obj'], dict['weight']
   45.1MB  (11008, 2048)                  torch.bfloat16       dict['obj'], dict['weight']
   45.1MB  (11008, 2048)                  torch.bfloat16       dict['obj'], dict['wei

## 6. MMLU — Dense baseline, BS 21%, Fused 21%

At 2B, dense baseline should be ~50% (well above random 25%) so pruning damage is measurable. We expect:
- BS 21% ≈ baseline - a few points (sparsity costs some accuracy)
- Fused 21% ≈ BS 21% (fused kernel should preserve model quality now that activation is correct)

In [13]:
from eval_mmlu import evaluate, print_results, save_results

# Dense baseline MMLU — reuse the shared model in-place
configure_model("dense")
print("=== Dense baseline MMLU ===")
t0 = time.time()
results_dense = evaluate(model, tokenizer, tag="qwen25_3b_dense")
results_dense["meta"] = {"model": MODEL_NAME_2B, "variant": "dense",
                         "eval_time_s": round(time.time() - t0, 1)}
print_results(results_dense)
save_results(results_dense)
mmlu_dense = results_dense["overall"]["accuracy"]


  configured 'dense' | pytorch allocated: 6.24 GB
=== Dense baseline MMLU ===


[qwen25_3b_dense] MMLU subjects: 100%|███████████████████████████████| 10/10 [01:22<00:00,  8.29s/it]


  MMLU Results — qwen25_3b_dense
  abstract_algebra               █████████░░░░░░░░░░░  45.0% (45/100)
  anatomy                        █████████████░░░░░░░  66.7% (90/135)
  college_chemistry              ████████░░░░░░░░░░░░  44.0% (44/100)
  college_computer_science       ████████████░░░░░░░░  60.0% (60/100)
  econometrics                   ██████████░░░░░░░░░░  50.0% (57/114)
  global_facts                   ████████░░░░░░░░░░░░  40.0% (40/100)
  machine_learning               █████████░░░░░░░░░░░  48.2% (54/112)
  moral_scenarios                ███████░░░░░░░░░░░░░  36.8% (329/895)
  professional_medicine          █████████████░░░░░░░  69.1% (188/272)
  us_foreign_policy              ████████████████░░░░  80.0% (80/100)
────────────────────────────────────────────────────────────
  OVERALL                                       48.7% (987/2028)

Results saved to results/mmlu_qwen25_3b_dense.json


In [14]:
# BS 21% MMLU — reconfigure in-place
configure_model("bs", prune_masks_21)
print("\n=== BS 21% MMLU ===")
t0 = time.time()
results_bs21 = evaluate(model, tokenizer, tag="qwen25_3b_bs_actw_21")
results_bs21["meta"] = {"model": MODEL_NAME_2B, "variant": "bs_21",
                        "eval_time_s": round(time.time() - t0, 1)}
print_results(results_bs21)
save_results(results_bs21)
mmlu_bs21 = results_bs21["overall"]["accuracy"]


  configured 'bs' | pytorch allocated: 6.24 GB

=== BS 21% MMLU ===


[qwen25_3b_bs_actw_21] MMLU subjects: 100%|██████████████████████████| 10/10 [01:22<00:00,  8.22s/it]


  MMLU Results — qwen25_3b_bs_actw_21
  abstract_algebra               █████░░░░░░░░░░░░░░░  25.0% (25/100)
  anatomy                        ███░░░░░░░░░░░░░░░░░  18.5% (25/135)
  college_chemistry              ████░░░░░░░░░░░░░░░░  23.0% (23/100)
  college_computer_science       ███████░░░░░░░░░░░░░  36.0% (36/100)
  econometrics                   ███░░░░░░░░░░░░░░░░░  18.4% (21/114)
  global_facts                   ███░░░░░░░░░░░░░░░░░  15.0% (15/100)
  machine_learning               ████░░░░░░░░░░░░░░░░  21.4% (24/112)
  moral_scenarios                ████░░░░░░░░░░░░░░░░  24.5% (219/895)
  professional_medicine          ██████░░░░░░░░░░░░░░  34.9% (95/272)
  us_foreign_policy              ████░░░░░░░░░░░░░░░░  21.0% (21/100)
────────────────────────────────────────────────────────────
  OVERALL                                       24.9% (504/2028)

Results saved to results/mmlu_qwen25_3b_bs_actw_21.json


In [15]:
# Fused 21% MMLU — reconfigure in-place
configure_model("fused", prune_masks_21)
print("\n=== Fused 21% MMLU ===")
t0 = time.time()
results_fused21 = evaluate(model, tokenizer, tag="qwen25_3b_fused_actw_21")
results_fused21["meta"] = {"model": MODEL_NAME_2B, "variant": "fused_21",
                           "eval_time_s": round(time.time() - t0, 1)}
print_results(results_fused21)
save_results(results_fused21)
mmlu_fused21 = results_fused21["overall"]["accuracy"]

print(f"\n{'='*60}")
print(f"{'Variant':<20} {'Accuracy':>10}  {'Delta vs dense':>15}")
print("-" * 60)
print(f"{'Dense':<20} {mmlu_dense*100:>9.1f}%")
print(f"{'BS 21%':<20} {mmlu_bs21*100:>9.1f}%  {(mmlu_bs21-mmlu_dense)*100:>+14.1f}pp")
print(f"{'Fused 21%':<20} {mmlu_fused21*100:>9.1f}%  {(mmlu_fused21-mmlu_dense)*100:>+14.1f}pp")
print("=" * 60)


  configured 'fused' | pytorch allocated: 6.24 GB

=== Fused 21% MMLU ===


[qwen25_3b_fused_actw_21] MMLU subjects: 100%|███████████████████████| 10/10 [01:15<00:00,  7.58s/it]


  MMLU Results — qwen25_3b_fused_actw_21
  abstract_algebra               █████░░░░░░░░░░░░░░░  25.0% (25/100)
  anatomy                        ███░░░░░░░░░░░░░░░░░  18.5% (25/135)
  college_chemistry              ████░░░░░░░░░░░░░░░░  23.0% (23/100)
  college_computer_science       ███████░░░░░░░░░░░░░  36.0% (36/100)
  econometrics                   ███░░░░░░░░░░░░░░░░░  17.5% (20/114)
  global_facts                   ███░░░░░░░░░░░░░░░░░  15.0% (15/100)
  machine_learning               ████░░░░░░░░░░░░░░░░  20.5% (23/112)
  moral_scenarios                ████░░░░░░░░░░░░░░░░  24.7% (221/895)
  professional_medicine          ███████░░░░░░░░░░░░░  36.8% (100/272)
  us_foreign_policy              ████░░░░░░░░░░░░░░░░  21.0% (21/100)
────────────────────────────────────────────────────────────
  OVERALL                                       25.1% (509/2028)

Results saved to results/mmlu_qwen25_3b_fused_actw_21.json

Variant                Accuracy   Delta vs dense
--------------------

In [16]:
# Fused 21% MMLU with Taylor/gradient masks — the comparison we care about
configure_model("fused", prune_masks_grad_21)
print("\n=== Fused 21% (Taylor) MMLU ===")
t0 = time.time()
results_fused_grad21 = evaluate(model, tokenizer, tag="qwen25_3b_fused_grad_21")
results_fused_grad21["meta"] = {"model": MODEL_NAME_2B, "variant": "fused_grad_21",
                                 "scoring": "gradient_taylor",
                                 "eval_time_s": round(time.time() - t0, 1)}
print_results(results_fused_grad21)
save_results(results_fused_grad21)
mmlu_fused_grad21 = results_fused_grad21["overall"]["accuracy"]

# Updated summary including Taylor
print(f"\n{'='*70}")
print(f"{'Variant':<30} {'Accuracy':>10}  {'Delta vs dense':>17}")
print("-" * 70)
print(f"{'Dense':<30} {mmlu_dense*100:>9.1f}%")
print(f"{'BS 21% (ActW)':<30} {mmlu_bs21*100:>9.1f}%  {(mmlu_bs21-mmlu_dense)*100:>+16.1f}pp")
print(f"{'Fused 21% (ActW)':<30} {mmlu_fused21*100:>9.1f}%  {(mmlu_fused21-mmlu_dense)*100:>+16.1f}pp")
print(f"{'Fused 21% (Taylor)':<30} {mmlu_fused_grad21*100:>9.1f}%  {(mmlu_fused_grad21-mmlu_dense)*100:>+16.1f}pp")
print("=" * 70)
print("Taylor vs ActW: compare the Fused 21% rows — both run the same kernel + sparsity,")
print("only the chosen tiles differ. Bigger accuracy on Taylor = better scoring method.")


  configured 'fused' | pytorch allocated: 6.24 GB

=== Fused 21% (Taylor) MMLU ===


[qwen25_3b_fused_grad_21] MMLU subjects: 100%|███████████████████████| 10/10 [01:17<00:00,  7.77s/it]


  MMLU Results — qwen25_3b_fused_grad_21
  abstract_algebra               ████░░░░░░░░░░░░░░░░  23.0% (23/100)
  anatomy                        █████░░░░░░░░░░░░░░░  27.4% (37/135)
  college_chemistry              █████░░░░░░░░░░░░░░░  29.0% (29/100)
  college_computer_science       ████░░░░░░░░░░░░░░░░  24.0% (24/100)
  econometrics                   █████░░░░░░░░░░░░░░░  26.3% (30/114)
  global_facts                   ███░░░░░░░░░░░░░░░░░  18.0% (18/100)
  machine_learning               █████░░░░░░░░░░░░░░░  25.0% (28/112)
  moral_scenarios                █████░░░░░░░░░░░░░░░  27.3% (244/895)
  professional_medicine          ███████░░░░░░░░░░░░░  36.8% (100/272)
  us_foreign_policy              ████░░░░░░░░░░░░░░░░  24.0% (24/100)
────────────────────────────────────────────────────────────
  OVERALL                                       27.5% (557/2028)

Results saved to results/mmlu_qwen25_3b_fused_grad_21.json

Variant                          Accuracy     Delta vs dense
--------

In [17]:
# Build Taylor-based 50% uniform masks (bottom 50% per matrix by Taylor score)
prune_masks_grad_50 = {}
for name, norm_map in importance_grad_norm.items():
    local = float(np.percentile(norm_map.ravel(), 50))
    prune_masks_grad_50[name] = norm_map < local
p_g50, t_g50, r_g50 = eff_sparsity(prune_masks_grad_50)
print(f"50% Taylor uniform: {p_g50}/{t_g50} = {100*r_g50:.1f}% effective")

# Fused 50% (Taylor) MMLU
configure_model("fused", prune_masks_grad_50)
print("\n=== Fused 50% (Taylor) MMLU ===")
t0 = time.time()
results_fused_grad50 = evaluate(model, tokenizer, tag="qwen25_3b_fused_grad_50")
results_fused_grad50["meta"] = {"model": MODEL_NAME_2B, "variant": "fused_grad_50",
                                 "scoring": "gradient_taylor",
                                 "eval_time_s": round(time.time() - t0, 1)}
print_results(results_fused_grad50)
save_results(results_fused_grad50)
mmlu_fused_grad50 = results_fused_grad50["overall"]["accuracy"]

# Full comparison table — all MMLU results
print(f"\n{'='*75}")
print(f"{'Variant':<35} {'Accuracy':>10}  {'Delta vs dense':>17}")
print("-" * 75)
print(f"{'Dense':<35} {mmlu_dense*100:>9.1f}%")
print(f"{'BS 21% (ActW)':<35} {mmlu_bs21*100:>9.1f}%  {(mmlu_bs21-mmlu_dense)*100:>+16.1f}pp")
print(f"{'Fused 21% (ActW)':<35} {mmlu_fused21*100:>9.1f}%  {(mmlu_fused21-mmlu_dense)*100:>+16.1f}pp")
print(f"{'Fused 21% (Taylor)':<35} {mmlu_fused_grad21*100:>9.1f}%  {(mmlu_fused_grad21-mmlu_dense)*100:>+16.1f}pp")
print(f"{'Fused 50% (Taylor)':<35} {mmlu_fused_grad50*100:>9.1f}%  {(mmlu_fused_grad50-mmlu_dense)*100:>+16.1f}pp")
print("=" * 75)
print("Comparison: At same 21% sparsity, Taylor beats ActW by {:.1f}pp.".format(
    (mmlu_fused_grad21 - mmlu_fused21)*100))
print("At 50%, Taylor shows how much accuracy we lose to push speedups higher.")


50% Taylor uniform: 74304/148608 = 50.0% effective
  configured 'fused' | pytorch allocated: 6.24 GB

=== Fused 50% (Taylor) MMLU ===


[qwen25_3b_fused_grad_50] MMLU subjects: 100%|███████████████████████| 10/10 [01:03<00:00,  6.39s/it]


  MMLU Results — qwen25_3b_fused_grad_50
  abstract_algebra               ███░░░░░░░░░░░░░░░░░  18.0% (18/100)
  anatomy                        ███░░░░░░░░░░░░░░░░░  18.5% (25/135)
  college_chemistry              ██████░░░░░░░░░░░░░░  34.0% (34/100)
  college_computer_science       ██████░░░░░░░░░░░░░░  31.0% (31/100)
  econometrics                   ████░░░░░░░░░░░░░░░░  24.6% (28/114)
  global_facts                   ████░░░░░░░░░░░░░░░░  20.0% (20/100)
  machine_learning               █████░░░░░░░░░░░░░░░  26.8% (30/112)
  moral_scenarios                ████░░░░░░░░░░░░░░░░  24.7% (221/895)
  professional_medicine          ████████░░░░░░░░░░░░  40.1% (109/272)
  us_foreign_policy              █████░░░░░░░░░░░░░░░  27.0% (27/100)
────────────────────────────────────────────────────────────
  OVERALL                                       26.8% (543/2028)

Results saved to results/mmlu_qwen25_3b_fused_grad_50.json

Variant                               Accuracy     Delta vs dense
---

## 7. End-to-end latency benchmarks (5 configs × 3 scenarios)

In [18]:
def bench_generate(model, inputs, max_new_tokens=50, n_warmup=2, n_iter=5):
    with torch.no_grad():
        for _ in range(n_warmup):
            model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iter):
            model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()
    return (time.time() - t0) / n_iter

def bench_prefill(model, inputs, n_warmup=3, n_iter=10):
    with torch.no_grad():
        for _ in range(n_warmup):
            model(**inputs)
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iter):
            model(**inputs)
        torch.cuda.synchronize()
    return (time.time() - t0) / n_iter

short_text = "Once upon a time"
long_text = " ".join(cal_texts[:10])
short_inputs = tokenizer(short_text, return_tensors="pt").to("cuda")
long_inputs  = tokenizer(long_text, return_tensors="pt", max_length=2000, truncation=True).to("cuda")
short_len = short_inputs["input_ids"].shape[1]
long_len  = long_inputs["input_ids"].shape[1]
print(f"short prompt: {short_len} tokens")
print(f"long prompt:  {long_len} tokens")

def bench_all(model, tag):
    s = bench_generate(model, short_inputs, max_new_tokens=50)
    l = bench_generate(model, long_inputs,  max_new_tokens=20)
    p = bench_prefill(model, long_inputs)
    print(f"[{tag:<12}] short: {s*1000:7.1f}ms | long: {l*1000:7.1f}ms | prefill: {p*1000:7.1f}ms | peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    return s, l, p

short prompt: 4 tokens
long prompt:  1915 tokens


In [19]:
# ============ End-to-end benchmarks across all 5 configs ============
print("--- Dense ---")
configure_model("dense")
d_short, d_long, d_prefill = bench_all(model, "Dense")

print("\n--- BS 21% ---")
configure_model("bs", prune_masks_21)
bs21_short, bs21_long, bs21_prefill = bench_all(model, "BS 21%")

print("\n--- BS 50% ---")
configure_model("bs", prune_masks_50)
bs50_short, bs50_long, bs50_prefill = bench_all(model, "BS 50%")

print("\n--- Fused 21% ---")
configure_model("fused", prune_masks_21)
f21_short, f21_long, f21_prefill = bench_all(model, "Fused 21%")

print("\n--- Fused 50% ---")
configure_model("fused", prune_masks_50)
f50_short, f50_long, f50_prefill = bench_all(model, "Fused 50%")

# Restore to dense state
configure_model("dense")


--- Dense ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  configured 'dense' | pytorch allocated: 6.24 GB
[Dense       ] short:  1422.7ms | long:   908.6ms | prefill:   385.2ms | peak GPU: 7.01 GB

--- BS 21% ---
  configured 'bs' | pytorch allocated: 6.24 GB
[BS 21%      ] short:  1423.6ms | long:   918.1ms | prefill:   396.7ms | peak GPU: 7.01 GB

--- BS 50% ---
  configured 'bs' | pytorch allocated: 6.24 GB
[BS 50%      ] short:  1423.4ms | long:   823.6ms | prefill:   301.5ms | peak GPU: 7.01 GB

--- Fused 21% ---
  configured 'fused' | pytorch allocated: 6.24 GB
[Fused 21%   ] short:  1423.4ms | long:   890.0ms | prefill:   368.6ms | peak GPU: 7.01 GB

--- Fused 50% ---
  configured 'fused' | pytorch allocated: 6.24 GB
[Fused 50%   ] short:  1423.9ms | long:   796.4ms | prefill:   274.2ms | peak GPU: 7.01 GB
  configured 'dense' | pytorch allocated: 6.24 GB


In [20]:
# === Comparison table ===
print(f"\n{'='*110}")
print(f"{'Scenario':<34} {'Dense':>10} {'BS 21%':>10} {'BS 50%':>10} {'Fused 21%':>11} {'Fused 50%':>11}")
print("-" * 110)
def row(label, d, b21, b50, f21, f50):
    print(f"{label:<34} "
          f"{d*1000:>8.1f}ms {b21*1000:>8.1f}ms {b50*1000:>8.1f}ms "
          f"{f21*1000:>9.1f}ms {f50*1000:>9.1f}ms")
row("short + 50 gen (decode)",   d_short,   bs21_short,   bs50_short,   f21_short,   f50_short)
row("long + 20 gen (mixed)",     d_long,    bs21_long,    bs50_long,    f21_long,    f50_long)
row("prefill only",              d_prefill, bs21_prefill, bs50_prefill, f21_prefill, f50_prefill)
print("=" * 110)

# Speedup vs dense
print(f"\nSpeedup vs dense (>1.0x = win):")
print(f"{'Scenario':<34} {'BS 21%':>9} {'BS 50%':>9} {'Fused 21%':>10} {'Fused 50%':>10}")
print("-" * 85)
def sp(label, d, b21, b50, f21, f50):
    print(f"{label:<34} {d/b21:>8.2f}x {d/b50:>8.2f}x {d/f21:>9.2f}x {d/f50:>9.2f}x")
sp("short + 50 gen",   d_short,   bs21_short,   bs50_short,   f21_short,   f50_short)
sp("long + 20 gen",    d_long,    bs21_long,    bs50_long,    f21_long,    f50_long)
sp("prefill only",     d_prefill, bs21_prefill, bs50_prefill, f21_prefill, f50_prefill)
print("=" * 85)

# Fusion gain
print(f"\nFusion gain (Fused / BS at same sparsity, >1.0x means fusion helped):")
print(f"{'Scenario':<34} {'Fused/BS 21%':>14} {'Fused/BS 50%':>14}")
print("-" * 70)
def fg(label, b21, f21, b50, f50):
    print(f"{label:<34} {b21/f21:>13.2f}x {b50/f50:>13.2f}x")
fg("short + 50 gen",   bs21_short,   f21_short,   bs50_short,   f50_short)
fg("long + 20 gen",    bs21_long,    f21_long,    bs50_long,    f50_long)
fg("prefill only",     bs21_prefill, f21_prefill, bs50_prefill, f50_prefill)
print("=" * 70)


Scenario                                Dense     BS 21%     BS 50%   Fused 21%   Fused 50%
--------------------------------------------------------------------------------------------------------------
short + 50 gen (decode)              1422.7ms   1423.6ms   1423.4ms    1423.4ms    1423.9ms
long + 20 gen (mixed)                 908.6ms    918.1ms    823.6ms     890.0ms     796.4ms
prefill only                          385.2ms    396.7ms    301.5ms     368.6ms     274.2ms

Speedup vs dense (>1.0x = win):
Scenario                              BS 21%    BS 50%  Fused 21%  Fused 50%
-------------------------------------------------------------------------------------
short + 50 gen                         1.00x     1.00x      1.00x      1.00x
long + 20 gen                          0.99x     1.10x      1.02x      1.14x
prefill only                           0.97x     1.28x      1.04x      1.40x

Fusion gain (Fused / BS at same sparsity, >1.0x means fusion helped):
Scenario              

## Summary

Expected outcomes:
- **MMLU dense baseline** ~50% (Gemma 2 2B on MMLU subset); BS 21% drops a few points; Fused 21% should match BS 21% now that activation is correct.
- **Prefill speedup** should be bigger than at 1B. Hypothesis: 50% sparsity prefill hits ~1.25–1.35x at 2B (vs 1.14x at 1B).
- **21% break-even**: at 1B it hovered near 1.0x; at 2B the bigger matmul should tip it clearly positive.
- **Fusion gain**: if the 2B regime makes the kernel matter more, `gate_proj` intermediate is `M×9216` (vs `M×6912`) so avoiding that round-trip saves more. Might finally show measurable fusion benefit.

If prefill speedups do scale as predicted, this validates that our approach has a ceiling we can push by going bigger, not a fundamental flaw.